In [ ]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt
import time
from __future__ import annotations
import json

In [ ]:
def jumping(data, lowerBound, upperBound):
    j = 0
    if(len(data) == len(lowerBound) and  len(data) == len(upperBound)):
        inj = []
        for i in range(len(data)):
            inj.append(0)
            
        for i in range(len(data)):    
            if(data[i] >= upperBound[i] and inj[i]==0):
                inj[i] = 1
                j = i - 1
                while(j >= 0):
                    if(data[j] > lowerBound[j]):
                        inj[j] = 1
                    else:
                        break
                    j -= 1
                j = i + 1
                while(j < len(data)):
                    if(data[j] > lowerBound[j]):
                        inj[j] = 1
                    else:
                        break
                    j += 1

        for i in range(len(data)):
            if(lowerBound[i] == upperBound[i]):
                inj[i] = 2;
    else:
        print("bouds are wrong")

    return inj

In [ ]:
def compareRightLeft(right, left):
    if(len(right) == len(left)):
        result = []
        for i in range(len(right)):
            result.append(0)
        
        for i in range(len(right)):
            result[i] = right[i];
            if (left[i] != 1):
                result[i] = 3;
            
    else:
        print("bounds are wrong")
    return result

In [ ]:
def countJumps(jumps):
    count = 0
    
    inJump = False
    for i in range(len(jumps)):
        if(jumps[i] == 3):
            inJump = True
        if(inJump and jumps[i-1] == 1 and jumps[i] == 0):
            inJump = False
            count += 1
    return count

In [ ]:
import math
def movingPercentile(data: list[float], window_size: int, percentile: float) -> list[float]:
    assert 0 <= percentile <= 1, "Percentile must be between 0 and 1"
    
    result = [0.0] * len(data)
    
    for i in range(len(data)):
        window_data = []
        for j in range(-window_size, window_size + 1):
            index = i + j
            if 0 <= index < len(data):
                if not math.isnan(data[index]):
                    window_data.append(data[index])
        
        window_data.sort()
        n = len(window_data)
        
        if n > 0:
            k = int(math.floor(percentile * (n - 1)))
            result[i] = window_data[k]
        else:
            result[i] = float('nan')
    
    return result

In [ ]:
def normalize(data):
    result = []
    for i in range(len(data)):
        result.append(0)
    
    dataMin = data[0]
    for i in range(len(data)):
        if(data[i] < dataMin):
            dataMin = data[i]
    for i in range(len(data)):
        result[i] = data[i] - dataMin
    return result

In [ ]:
def calcBound(arrMin, arrMax, thr):
    result = []
    for i in range(len(arrMin)):
        result.append(arrMin[i] + (arrMax[i] - arrMin[i]) * thr)
    return result

In [ ]:
def twoLegCounting(
    kneeYRKneeInput: Sequence[float],
    kneeYLKneeInput: Sequence[float],               
    fps: float):
    
    windowSize = fps 
    
    YLKnee = [-x for x in kneeYLKneeInput]
    YLKnee = normalize(YLKnee)
    YRKnee = [-x for x in kneeYRKneeInput]
    YRKnee = normalize(YRKnee)

    lb = 0.1
    ub = 0.9
    arrP20R = movingPercentile(YRKnee, int(windowSize), lb)
    arrP20L = movingPercentile(YLKnee, int(windowSize), lb)
    arrP80R = movingPercentile(YRKnee, int(windowSize), ub)
    arrP80L = movingPercentile(YLKnee, int(windowSize), ub)

    rLB = calcBound(arrP20R,arrP80R, 0.3)
    rUB = calcBound(arrP20R,arrP80R, 0.7)
    lLB = calcBound(arrP20L,arrP80L, 0.3)
    lUB = calcBound(arrP20L,arrP80L, 0.7)
    
    arrJumpingR = jumping(YRKnee, rLB, rUB)
    arrJumpingL = jumping(YLKnee, lLB, lUB)
    
    arrJumpingRL = compareRightLeft(arrJumpingR, arrJumpingL)
    count = countJumps(arrJumpingRL)
    print("Jump count: ", count)
    

In [ ]:
testInputRight = [563, 563, 563, 563, 540, 540, 516, 516, 493, 493, 516, 540, 540, 540, 540, 540, 516, 516, 493, 493, 469, 469, 493, 493, 516, 516, 540, 540, 540, 516, 493, 493, 469, 469, 493, 493, 516, 540, 540, 540, 540, 516, 493, 493, 469, 493, 493, 516, 540, 540, 540, 540, 516, 493, 493, 469, 469, 493, 493, 516, 540, 540, 540, 540, 516, 493, 493, 469, 469, 493, 493, 516, 540, 540, 540, 516, 493, 493, 493, 469, 469, 493, 516, 516, 540, 540, 540, 516, 493, 493, 469, 469, 469, 493, 516, 516, 540, 540, 540, 516, 516, 493, 493, 469, 469, 493, 493, 516, 540, 540, 540, 516, 516, 493, 469, 469, 469, 493, 493, 516, 540, 540, 540, 516, 516, 493, 493, 469, 469, 493, 516, 516, 540, 540, 540, 516, 516, 493, 493]
testInputLeft =[563, 563, 563, 563, 540, 563, 563, 540, 540, 516, 516, 493, 493, 493, 469, 493, 493, 493, 516, 540, 540, 540, 516, 493, 493, 493, 469, 469, 493, 493, 516, 540, 563, 540, 516, 493, 493, 493, 469, 493, 493, 493, 516, 540, 540, 540, 516, 493, 493, 469, 469, 493, 493, 516, 540, 540, 540, 516, 493, 493, 493, 469, 469, 493, 493, 516, 540, 540, 540, 516, 493, 493, 493, 469, 469, 493, 493, 516, 540, 540, 516, 493, 493, 493, 493, 469, 493, 493, 493, 516, 540, 540, 516, 516, 493, 493, 469, 469, 469, 493, 516, 516, 540, 540, 540, 516, 493, 493, 469, 469, 469, 493, 516, 516, 540, 540, 540, 516, 493, 493, 469, 469, 469, 469, 493, 516, 540, 540, 540, 516, 493, 493, 469, 469, 469, 469, 493, 516, 540]
testInputSlowRight = [493, 493, 493, 493, 493, 493, 493, 493, 493, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 446, 469, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 469, 469, 469, 469, 469, 469, 469, 469, 493, 469, 469, 493, 493, 493, 469, 493, 493, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 446, 446, 446, 446, 446, 469, 446, 469, 446, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 493, 493, 469, 469, 493, 493, 493, 493, 469, 469, 493, 493, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 446, 446, 446, 446, 446, 469, 469, 469, 469, 446, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 493, 493, 493, 493, 493, 493, 493, 493, 469, 493, 493, 469, 493, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 446, 469, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 446, 446, 446, 446, 446, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 493, 469, 493, 493, 493, 493, 493, 493, 493, 493, 493, 493, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 446, 446, 446, 446, 446, 446, 446, 446, 446, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 493, 469, 469, 493, 469, 469, 493, 493, 493, 493, 493, 493, 493, 493, 493, 493, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 446, 422, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 493, 469, 469, 469, 493, 493, 493, 493, 493, 493, 469, 493, 493, 493, 493, 493, 493, 493, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 446, 422, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 493, 469, 493, 493, 493, 493, 493, 493, 493, 493, 493, 493, 493, 493, 493, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 422, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 469, 469, 469, 469, 469, 469, 469, 469, 469, 493, 493]
testInputSlowLeft = [446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 469, 446, 446, 469, 469, 469, 469, 446, 469, 469, 469, 469, 469, 469, 469, 469, 469, 493, 493, 493, 493, 493, 493, 493, 493, 493, 493, 493, 493, 493, 493, 493, 493, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 446, 446, 446, 446, 469, 469, 469, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 493, 469, 469, 469, 493, 493, 493, 493, 493, 493, 493, 493, 493, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 446, 446, 446, 469, 446, 446, 469, 469, 469, 446, 469, 446, 469, 469, 446, 469, 469, 446, 446, 469, 469, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 469, 446, 446, 446, 446, 446, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 493, 493, 469, 493, 493, 493, 493, 493, 493, 493, 493, 493, 493, 493, 493, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 446, 446, 446, 469, 446, 446, 469, 446, 446, 469, 469, 446, 446, 446, 446, 446, 469, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 493, 469, 493, 493, 469, 469, 493, 493, 493, 493, 493, 493, 493, 493, 493, 469, 493, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 446, 469, 469, 446, 469, 469, 469, 469, 469, 446, 446, 469, 469, 446, 469, 446, 469, 446, 469, 469, 446, 469, 469, 446, 469, 469, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 469, 446, 446, 469, 446, 446, 446, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 493, 493, 493, 493, 493, 469, 493, 493, 493, 493, 493, 493, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 446, 469, 469, 446, 469, 469, 446, 469, 469, 469, 469, 469, 469, 446, 446, 446, 446, 469, 446, 469, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 469, 446, 446, 446, 446, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 493, 493, 469, 469, 493, 493, 493, 493, 493, 493, 493, 493, 493, 493, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 446, 469, 446, 446, 446, 446, 446, 446, 469, 446, 446, 469, 469, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 493, 469, 493, 469, 493, 493, 493, 493, 493, 493, 493, 493, 493, 493, 493, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 446, 446, 446, 469, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 446, 469, 469, 446, 446, 446, 446, 469, 469, 446, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 493, 469, 493, 493, 493, 493, 493, 493, 493, 493, 493, 493, 493, 469, 493, 493, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 469, 446, 469, 446, 446, 469, 446, 446, 469, 469, 446, 446, 469, 469, 469, 469, 469, 469, 446, 446, 446, 469, 469]
testingR = 
testingL = 
twoLegCounting(testingR, testingL, 30)

In [ ]:


protoFile = "pose_deploy_linevecCOCO.prototxt"
weightsFile = "pose_iter_440000.caffemodel"
video_source = "" 
fps = 30
threshold = 0.3
inWidth = 368
inHeight = 368
font = cv.FONT_HERSHEY_SIMPLEX

frame_data = []

BODY_PARTS = { "Nose": 0, "Neck": 1, "RShoulder": 2, "RElbow": 3, "RWrist": 4,
               "LShoulder": 5, "LElbow": 6, "LWrist": 7, "RHip": 8, "RKnee": 9,
               "RAnkle": 10, "LHip": 11, "LKnee": 12, "LAnkle": 13, "REye": 14,
               "LEye": 15, "REar": 16, "LEar": 17, "Background": 18 }

TRACKED_PARTS = [8, 9, 10, 11, 12, 13]

SELECTED_POSE_PAIRS = [["RHip", "RKnee"], ["RKnee", "RAnkle"], ["LHip", "LKnee"], ["LKnee", "LAnkle"]]

net = cv.dnn.readNetFromCaffe(protoFile, weightsFile)
cap = cv.VideoCapture(video_source)

print("Video Start; q zum stoppen")

outputYRKnee = []
outputYLKnee = []
conf_list = []
missing_frames = 0
frames = 0
total_start_time = time.time()

try:
    while True:
        hasFrame, frame = cap.read()
        if not hasFrame:
            break
        
        frame_start_time = time.time() # Startzeit für diesen Frame
        
        frameWidth = frame.shape[1]
        frameHeight = frame.shape[0]
        frames += 1

        current_frame_info = {"frame": frames, "RKnee": None, "LKnee": None}
        
        #Verarbeitung durch KI
        inp = cv.dnn.blobFromImage(frame, 1.0 / 255, (inWidth, inHeight),
                                  (0, 0, 0), swapRB=False, crop=False)
        net.setInput(inp)
        out = net.forward()

        #Koordinaten der keypoints
        points = []
        for i in range(len(BODY_PARTS)):
            heatMap = out[0, i, :, :]
            _, conf, _, point = cv.minMaxLoc(heatMap)
            x = (frameWidth * point[0]) / out.shape[3]
            y = (frameHeight * point[1]) / out.shape[2]
            points.append((int(x), int(y)) if conf > threshold else None)

            if i == 9: #if RKnee
                outputYRKnee.append((int(y)) if conf > threshold else 0)
                conf_list.append(conf)
                if conf <= threshold:
                    missing_frames += 1
                current_frame_info["RKnee"] = {"x": int(x), "y": int(y)}
            if i == 12: 
                outputYLKnee.append((int(y)) if conf > threshold else 0)
                current_frame_info["LKnee"] = {"x": int(x), "y": int(y)}

        frame_data.append(current_frame_info)
        
        #Visualisieren (debug)
        for pair in SELECTED_POSE_PAIRS:
            partFrom, partTo = pair[0], pair[1]
            idFrom, idTo = BODY_PARTS[partFrom], BODY_PARTS[partTo]
            if points[idFrom] and points[idTo]:
                cv.line(frame, points[idFrom], points[idTo], (0, 255, 0), 3)
                cv.circle(frame, points[idFrom], 5, (0, 0, 255), -1)
                cv.circle(frame, points[idTo], 5, (0, 0, 255), -1)

        #frame output
        elapsed_time = time.time() - frame_start_time
        fps = 1 / elapsed_time if elapsed_time > 0 else 0
        
        print(f"Verarbeite Frame: {frames} | FPS: {fps:.2f} | Dauer: {elapsed_time:.3f}s", end='\r')
        
        cv.imshow('OpenPose Video', frame)
        if cv.waitKey(1) & 0xFF == ord('q'):
            break

finally:
    #with open('test2v3Output.json', 'w') as f:
    #    json.dump(frame_data, f, indent=4)
    
    total_duration = time.time() - total_start_time
    print("\n" + "-"*30) # Neue Zeile nach der \r Ausgabe
    if conf_list:
        avg_conf = sum(conf_list) / len(conf_list)
        print(f"Durchschnittliche Confidence: {avg_conf:.5f}")
    
    print(f"Gesamtzeit: {total_duration:.2f} Sekunden")
    print(f"Frames verarbeitet: {frames}") 
    print(f"Fehlende Frames (RKnee): {missing_frames}")
    print("-"*30)

    print("Right knee output: ", outputYRKnee)
    print("Left knee output: ", outputYLKnee)
    
    cap.release()
    cv.destroyAllWindows()
    print("Video window closed.")